In [44]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc, sum, avg,udf
from pyspark.sql.types import StringType,IntegerType

# Initialize
spark = SparkSession.builder.appName("SparkCompleteNotes").getOrCreate()
# Create Base DataFrame
data = [
    (101, "Aarav", "HR", 45000),
    (102, "Diya", "IT", 60000),
    (103, "Vivaan", "Finance", 55000),
    (104, "Anaya", "Marketing", 50000),
    (105, "Aditya", "IT", 65000),
    (106, "Ishita", "HR", 47000),
    (107, "Krishna", "Sales", 52000),
    (108, "Meera", "Finance", 58000),
    (109, "Arjun", "IT", 70000),
    (110, "Riya", "Marketing", 49000),
    (111, "Kabir", "Sales", 54000),
    (112, "Saanvi", "HR", 46000),
    (113, "Reyansh", "IT", 72000),
    (114, "Aadhya", "Finance", 61000),
    (115, "Atharv", "Sales", 53000),
    (116, "Myra", "Marketing", 51000),
    (117, "Shaurya", "IT", 68000),
    (118, "Pari", "HR", 48000),
    (119, "Yash", "Finance", 59000),
    (120, "Kiara", "Sales", 56000)
]
columns = ["Id", "Name", "Department", "Salary"]
df = spark.createDataFrame(data, columns);

In [2]:
df.write.mode("overwrite").csv("output/employee_data",header = True)

26/06/13 04:27:13 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

In [3]:
df.toPandas().to_csv(
    "output/employeetable2.csv",
    index = False
)

In [12]:
df = spark.range(0,10000000).withColumn("value",col("id")%1000)

In [13]:
df.take(10)

[Row(id=0, value=0),
 Row(id=1, value=1),
 Row(id=2, value=2),
 Row(id=3, value=3),
 Row(id=4, value=4),
 Row(id=5, value=5),
 Row(id=6, value=6),
 Row(id=7, value=7),
 Row(id=8, value=8),
 Row(id=9, value=9)]

In [4]:
df.take(10)

[Row(Id=101, Name='Aarav', Department='HR', Salary=45000),
 Row(Id=102, Name='Diya', Department='IT', Salary=60000),
 Row(Id=103, Name='Vivaan', Department='Finance', Salary=55000),
 Row(Id=104, Name='Anaya', Department='Marketing', Salary=50000),
 Row(Id=105, Name='Aditya', Department='IT', Salary=65000),
 Row(Id=106, Name='Ishita', Department='HR', Salary=47000),
 Row(Id=107, Name='Krishna', Department='Sales', Salary=52000),
 Row(Id=108, Name='Meera', Department='Finance', Salary=58000),
 Row(Id=109, Name='Arjun', Department='IT', Salary=70000),
 Row(Id=110, Name='Riya', Department='Marketing', Salary=49000)]

In [5]:
print("Before Partitions:",df.rdd.getNumPartitions())

Before Partitions: 12


In [6]:
df_repartitioned = df.repartition(10)

In [7]:
print("After Partitions:",df_repartitioned.rdd.getNumPartitions())

[Stage 5:=============================>                            (6 + 6) / 12]

After Partitions: 10


In [8]:
df_coalesced = df_repartitioned.coalesce(2)

In [9]:
print("After Coalesced:",df_coalesced.rdd.getNumPartitions())

After Coalesced: 2


[Stage 6:========================>                                 (5 + 7) / 12]

In [10]:
df_repartitioned.write.mode("overwrite").csv("output/employee_data_afterPartitioned",header=True)

In [11]:
df_coalesced.write.mode("overwrite").csv("output/employee_data_coalesced",header=True)

In [ ]:
# Optimization and caching

In [26]:
optimized_df = df.filter(col("value")>500).filter(col("id")<5000000).select("id","value")

In [27]:
optimized_df.explain()

== Physical Plan ==
*(1) Project [id#20L, (id#20L % 1000) AS value#22L]
+- *(1) Filter (((id#20L % 1000) > 500) AND (id#20L < 5000000))
   +- *(1) Range (0, 10000000, step=1, splits=12)




In [28]:
import time
start_time = time.time()
count_uncached = optimized_df.count() 
# Action triggers DAG
end_time = time.time()
print(f"1. Optimized Execution | Count: {count_uncached} | Time: {round(end_time-start_time,4)}seconds")

1. Optimized Execution | Count: 2495000 | Time: 0.1857seconds


In [29]:
optimized_df.cache()

DataFrame[id: bigint, value: bigint]

In [33]:
start_time = time.time()
count_first_cache = optimized_df.count() 
# Action triggers DAG
end_time = time.time()
print(f"2. First Cached Action | Count: {count_first_cache} | Time: {round(end_time-start_time,4)}seconds (Materializing cache)")

2. First Cached Action | Count: 2495000 | Time: 0.1614seconds (Materializing cache)


In [36]:
start_time = time.time()
count_second_cache = optimized_df.count() 
# Action triggers DAG
end_time = time.time()
print(f"3. Second Cached Action | Count: {count_second_cache} | Time: {round(end_time-start_time,4)}seconds (Read from memory)")

3. Second Cached Action | Count: 2495000 | Time: 0.1968seconds (Read from memory)


In [35]:
optimized_df.unpersist()
print("\n--> Unpersisted dataframe to free up memory.")


--> Unpersisted dataframe to free up memory.


In [ ]:
# udfs

In [37]:
data = [("Alice",25),("Bob",17),("Charlie",25)]
columns = ["Name","Age"]
df1 = spark.createDataFrame(data, columns);

In [38]:
df1.show()

+-------+---+
|   Name|Age|
+-------+---+
|  Alice| 25|
|    Bob| 17|
|Charlie| 25|
+-------+---+



In [39]:
def categorize_age(age):
    if age>=18:
        return "Adult"
    return "Minor"

In [42]:
age_category_udf = udf(categorize_age,StringType())

In [43]:
df_method1 = df1.withColumn("Category",age_category_udf(col("Age")))
print("Method 1: Standard UDF via DataFrame API")
df_method1.show()

Method 1: Standard UDF via DataFrame API


+-------+---+--------+
|   Name|Age|Category|
+-------+---+--------+
|  Alice| 25|   Adult|
|    Bob| 17|   Minor|
|Charlie| 25|   Adult|
+-------+---+--------+



In [45]:
# for sql

In [46]:
spark.udf.register("sql_categorize_age",categorize_age,StringType())

df1.createOrReplaceTempView("people")

In [47]:
sql_df = spark.sql("""
      SELECT
          Name,
          Age,
          sql_categorize_age(Age) AS Category
      FROM people
""")
sql_df.show()

+-------+---+--------+
|   Name|Age|Category|
+-------+---+--------+
|  Alice| 25|   Adult|
|    Bob| 17|   Minor|
|Charlie| 25|   Adult|
+-------+---+--------+

